In [27]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [29]:
## Load the file
data = pd.read_csv("churn_Modelling_synthetic.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15702874,Singh,665,Spain,Female,21,10,0.00,1,1,0,26844.25,0
1,2,15579292,Rossi,736,France,Female,35,9,0.00,2,0,0,191238.61,0
2,3,15688188,Martin,661,Spain,Male,22,4,0.00,1,1,1,176781.64,0
3,4,15678789,Smith,842,Germany,Male,59,9,110116.62,2,1,0,159572.29,1
4,5,15596014,Smith,651,France,Male,41,0,0.00,2,1,1,188105.13,0


In [30]:
## Preprocess the data
### Drop irrelevant cloumn
data = data.drop(["RowNumber","CustomerId","Surname"],axis=1)
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,665,Spain,Female,21,10,0.00,1,1,0,26844.25,0
1,736,France,Female,35,9,0.00,2,0,0,191238.61,0
2,661,Spain,Male,22,4,0.00,1,1,1,176781.64,0
3,842,Germany,Male,59,9,110116.62,2,1,0,159572.29,1
4,651,France,Male,41,0,0.00,2,1,1,188105.13,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,640,France,Female,53,3,151142.87,2,1,1,97003.21,1
9996,827,France,Male,45,5,0.00,2,1,0,127351.15,1
9997,520,France,Female,57,7,67446.29,1,0,0,56066.13,1
9998,650,France,Female,45,5,113426.44,1,0,0,83223.02,0


In [31]:
## Encode categorical variables
label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,665,Spain,0,21,10,0.00,1,1,0,26844.25,0
1,736,France,0,35,9,0.00,2,0,0,191238.61,0
2,661,Spain,1,22,4,0.00,1,1,1,176781.64,0
3,842,Germany,1,59,9,110116.62,2,1,0,159572.29,1
4,651,France,1,41,0,0.00,2,1,1,188105.13,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,640,France,0,53,3,151142.87,2,1,1,97003.21,1
9996,827,France,1,45,5,0.00,2,1,0,127351.15,1
9997,520,France,0,57,7,67446.29,1,0,0,56066.13,1
9998,650,France,0,45,5,113426.44,1,0,0,83223.02,0


In [32]:
## Onehot encode "Geography"
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder()
geo_encoder = onehot_encoder.fit_transform(data[["Geography"]])

In [33]:
onehot_encoder.get_feature_names_out(["Geography"])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [34]:
geo_encoaded_df = pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder.get_feature_names_out(["Geography"]))
geo_encoaded_df

,Geography_France,Geography_Germany,Geography_Spain
0,0.0,0.0,1.0
1,1.0,0.0,0.0
2,0.0,0.0,1.0
3,0.0,1.0,0.0
4,1.0,0.0,0.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,1.0,0.0,0.0


In [35]:
data = pd.concat([data.drop("Geography",axis=1),geo_encoaded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,665,0,21,10,0.00,1,1,0,26844.25,0,0.0,0.0,1.0
1,736,0,35,9,0.00,2,0,0,191238.61,0,1.0,0.0,0.0
2,661,1,22,4,0.00,1,1,1,176781.64,0,0.0,0.0,1.0
3,842,1,59,9,110116.62,2,1,0,159572.29,1,0.0,1.0,0.0
4,651,1,41,0,0.00,2,1,1,188105.13,0,1.0,0.0,0.0


In [36]:
## Save the encoders and scaler
with open("label_encoder_gender.pkl","wb") as file:
    pickle.dump(label_encoder_gender,file)

with open("onehot_encoder.pkl","wb") as file:
    pickle.dump(onehot_encoder,file)

In [37]:
## Divide the dataset into independent and dependent faetures

X = data.drop("Exited",axis=1)
y = data["Exited"]

## Split the data in training and testing sets
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## Scale these features 
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [38]:
X_test

array([[ 1.17275838,  0.91347777, -0.84388937, ...,  1.00828431,
        -0.58966846, -0.57138416],
       [-0.95555322,  0.91347777, -0.54160515, ..., -0.99178375,
        -0.58966846,  1.75013603],
       [ 1.13102678,  0.91347777, -0.44084374, ...,  1.00828431,
        -0.58966846, -0.57138416],
       ...,
       [ 1.38141638,  0.91347777, -0.54160515, ...,  1.00828431,
        -0.58966846, -0.57138416],
       [ 0.00427358, -1.09471739, -0.13855953, ...,  1.00828431,
        -0.58966846, -0.57138416],
       [ 0.08773678,  0.91347777,  0.66753172, ..., -0.99178375,
        -0.58966846,  1.75013603]])

In [39]:
import datetime

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential


In [40]:
model = Sequential([
    Dense(64,activation="relu",input_shape=(X_train.shape[1],)), # First HL1 connected with input
    Dense(32,activation="relu"), # Second HL2 
    Dense(1,"sigmoid")
])

In [18]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [41]:
import tensorflow
opt = tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss = tensorflow.keras.losses.BinaryCrossentropy()

In [42]:
# Compile
model.compile(optimizer=opt,loss=loss,metrics=["accuracy"])

In [43]:
## Set up the Tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard

log_dir = "logs/fit/"+ datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)


In [44]:
# Set up Early Stopping
early_stopping_callback = EarlyStopping(monitor="val_los",patience=5,restore_best_weights=True)

In [45]:
### Train the model
history = model.fit(
    X_train,y_train,validation_data=(X_test,y_test),epochs=100,
    callbacks=[tensorflow_callback,early_stopping_callback]
)

Epoch 1/100
250/250 [==============================] - 14s 34ms/step - loss: 0.4926 - accuracy: 0.7729 - val_loss: 0.4761 - val_accuracy: 0.7940
Epoch 2/100
250/250 [==============================] - 3s 11ms/step - loss: 0.4778 - accuracy: 0.7807 - val_loss: 0.4729 - val_accuracy: 0.7960
Epoch 3/100
250/250 [==============================] - 3s 11ms/step - loss: 0.4773 - accuracy: 0.7832 - val_loss: 0.4681 - val_accuracy: 0.7955
Epoch 4/100
250/250 [==============================] - 2s 9ms/step - loss: 0.4745 - accuracy: 0.7843 - val_loss: 0.4652 - val_accuracy: 0.8000
Epoch 5/100
250/250 [==============================] - 2s 7ms/step - loss: 0.4716 - accuracy: 0.7829 - val_loss: 0.4717 - val_accuracy: 0.7965
Epoch 6/100
250/250 [==============================] - 4s 17ms/step - loss: 0.4716 - accuracy: 0.7851 - val_loss: 0.4656 - val_accuracy: 0.7970
Epoch 7/100
250/250 [==============================] - 4s 17ms/step - loss: 0.4700 - accuracy: 0.7876 - val_loss: 0.4658 - val_accuracy: 

In [46]:
model.save("model.h5")

d:\AI Projects\first_projects\.venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [49]:
# Load Tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [50]:
%tensorboard -- logdir logs/fit

Launching TensorBoard...